# 🤖 Módulo 9: El Agente Autónomo — Reinforcement Learning
## Notebook de Conocimiento Guiado

**Objetivo:** Implementar Q-Learning desde cero, entender el Markov Decision Process
y entrenar un agente que aprende a navegar un GridWorld sin supervisión.

**Lore:** Eres un explorador IA enviado a un planeta alienígena (cuadrícula NxN).
Aprende a llegar al punto de extracción evitando las trampas de energía.

| Sección | Concepto |
|---------|---------|
| 📚 Teoría | MDP, Bellman, Q-Learning, epsilon-greedy |
| 🔨 Demo | GridWorldEnv + render ASCII |
| 🔨 Demo | Q-Agent con entrenamiento completo |
| ✏️ Ejercicio 1 | Double Q-Learning (reduce overestimation) |
| ✏️ Ejercicio 2 | Política epsilon-decay automático |
| 🎯 Quiz | RL, exploración vs explotación |

## 📚 Parte 1: El Marco MDP

Un **Markov Decision Process** tiene 4 componentes:

```
S — Espacio de estados    (cada celda de la cuadrícula)
A — Espacio de acciones   (arriba, abajo, izquierda, derecha)
R — Función de recompensa R(s, a, s') 
P — Función de transición P(s' | s, a)  ← determinista en nuestro caso
```

**Propiedad de Markov:** el siguiente estado solo depende del estado ACTUAL,
no de la historia. Cada celda es un estado completo.

### La Ecuación de Bellman

```
Q*(s,a) = R(s,a) + γ · max_{a'} Q*(s', a')
             ↑             ↑          ↑
          recompensa    descuento  mejor valor futuro
```

- **γ (gamma)** = factor de descuento (0.9): las recompensas futuras valen menos
- **Q*(s,a)** = valor óptimo de tomar acción `a` en estado `s`

### Q-Learning (actualización por TD)

```python
Q[s][a] ← Q[s][a] + α · (R + γ·max Q[s'][a'] − Q[s][a])
                      ↑               ↑
                   learning rate    target
```

In [ ]:
# 🔨 IMPLEMENTACIÓN: GridWorld Environment

import random

# Tipos de celda
VACIA, PARED, TRAMPA, META, INICIO = 0, 1, 2, 3, 4
ARRIBA, DERECHA, ABAJO, IZQUIERDA = 0, 1, 2, 3
DELTAS = {ARRIBA: (-1,0), DERECHA: (0,1), ABAJO: (1,0), IZQUIERDA: (0,-1)}
SIMBOLOS_ACCION = {ARRIBA: "↑", DERECHA: "→", ABAJO: "↓", IZQUIERDA: "←"}

RECOMPENSA_META = 100.0
RECOMPENSA_TRAMPA = -100.0
RECOMPENSA_PASO = -1.0

class GridWorldEnv:
    """
    Entorno GridWorld: cuadrícula NxN con meta, trampas y paredes.
    Sigue el patrón reset/step de la API de Gymnasium.
    """
    def __init__(self, n=5, semilla=42):
        random.seed(semilla)
        self.n = n
        self.grid = self._crear_grid()
        self.estado = None
        self.pasos = 0
    
    def _crear_grid(self):
        grid = [[VACIA]*self.n for _ in range(self.n)]
        grid[0][0] = INICIO
        grid[self.n-1][self.n-1] = META
        # Añadir trampas
        for fila, col in [(1,1),(2,3),(3,1),(0,3)]:
            if fila < self.n and col < self.n:
                grid[fila][col] = TRAMPA
        return grid
    
    def reset(self):
        self.estado = (0, 0)
        self.pasos = 0
        return self.estado
    
    def step(self, accion):
        fila, col = self.estado
        df, dc = DELTAS[accion]
        nueva_fila = max(0, min(self.n-1, fila + df))
        nueva_col  = max(0, min(self.n-1, col  + dc))
        
        tipo = self.grid[nueva_fila][nueva_col]
        if tipo == PARED:
            nueva_fila, nueva_col = fila, col   # Rebota
        
        self.estado = (nueva_fila, nueva_col)
        self.pasos += 1
        
        recompensa = (RECOMPENSA_META   if tipo == META    else
                      RECOMPENSA_TRAMPA if tipo == TRAMPA  else
                      RECOMPENSA_PASO)
        terminado = tipo in (META, TRAMPA) or self.pasos >= 200
        return self.estado, recompensa, terminado
    
    def render(self):
        iconos = {VACIA:"·", PARED:"█", TRAMPA:"☠", META:"★", INICIO:"S"}
        for i, fila in enumerate(self.grid):
            fila_str = ""
            for j, celda in enumerate(fila):
                if (i,j) == self.estado:
                    fila_str += "A "
                else:
                    fila_str += iconos[celda] + " "
            print(fila_str)
        print()

env = GridWorldEnv(n=5)
estado = env.reset()
print("Entorno inicial (A=agente, ★=meta, ☠=trampa):")
env.render()
print(f"Estado inicial: {estado}")

In [ ]:
# 🔨 IMPLEMENTACIÓN: Q-Learning Agent

import math

class QAgent:
    """
    Agente Q-Learning con política epsilon-greedy.
    
    La Q-table es un diccionario {(fila,col): [Q_arriba, Q_derecha, Q_abajo, Q_izquierda]}
    """
    def __init__(self, n_acciones=4, alpha=0.1, gamma=0.9, epsilon=0.5):
        self.n_acciones = n_acciones
        self.alpha = alpha         # Tasa de aprendizaje
        self.gamma = gamma         # Factor de descuento
        self.epsilon = epsilon     # Probabilidad de exploración
        self.q_table = {}          # (fila, col) → [Q(a0), Q(a1), Q(a2), Q(a3)]
    
    def _get_q(self, estado):
        if estado not in self.q_table:
            self.q_table[estado] = [0.0] * self.n_acciones
        return self.q_table[estado]
    
    def seleccionar_accion(self, estado):
        """Epsilon-greedy: explorar con prob epsilon, explotar con 1-epsilon."""
        if random.random() < self.epsilon:
            return random.randrange(self.n_acciones)   # Explorar
        q_vals = self._get_q(estado)
        return q_vals.index(max(q_vals))               # Explotar
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Actualiza la Q-table con la ecuación de Bellman."""
        q_actual = self._get_q(estado)[accion]
        max_q_siguiente = 0.0 if terminado else max(self._get_q(siguiente_estado))
        
        # Bellman update
        target = recompensa + self.gamma * max_q_siguiente
        self._get_q(estado)[accion] += self.alpha * (target - q_actual)

def entrenar(env, agente, episodios=500):
    """Entrena el agente en el entorno."""
    recompensas_totales = []
    for ep in range(episodios):
        estado = env.reset()
        recompensa_total = 0.0
        terminado = False
        
        while not terminado:
            accion = agente.seleccionar_accion(estado)
            siguiente, recompensa, terminado = env.step(accion)
            agente.aprender(estado, accion, recompensa, siguiente, terminado)
            estado = siguiente
            recompensa_total += recompensa
        
        # Decaer epsilon gradualmente
        agente.epsilon = max(0.01, agente.epsilon * 0.995)
        recompensas_totales.append(recompensa_total)
    
    return recompensas_totales

# Entrenar
env = GridWorldEnv(n=5, semilla=42)
agente = QAgent(alpha=0.1, gamma=0.9, epsilon=0.8)
recompensas = entrenar(env, agente, episodios=1000)

# Mostrar progreso
print("Progreso del entrenamiento:")
for bloque in range(0, 1000, 200):
    prom = sum(recompensas[bloque:bloque+200]) / 200
    barra = "█" * int((prom + 100) / 20)
    print(f"  Eps {bloque:4d}-{bloque+200}: {prom:7.1f} {barra}")

print(f"\nÚltimo epsilon: {agente.epsilon:.4f}")

In [ ]:
# 🔨 Visualizar la política aprendida

def mostrar_politica(env, agente):
    """Muestra la mejor acción para cada celda de la cuadrícula."""
    iconos_celda = {VACIA:"·", PARED:"█", TRAMPA:"☠", META:"★", INICIO:"S"}
    print("Política aprendida (↑↓←→ = mejor acción):")
    for i in range(env.n):
        fila_str = ""
        for j in range(env.n):
            tipo = env.grid[i][j]
            if tipo in (TRAMPA, META, PARED):
                fila_str += iconos_celda[tipo] + " "
            else:
                q_vals = agente._get_q((i, j))
                mejor = q_vals.index(max(q_vals))
                fila_str += SIMBOLOS_ACCION[mejor] + " "
        print("  " + fila_str)

mostrar_politica(env, agente)

# Demostrar episodio con política aprendida
agente.epsilon = 0.0   # Sin exploración
env2 = GridWorldEnv(n=5, semilla=42)
estado = env2.reset()
ruta = [estado]
for _ in range(30):
    accion = agente.seleccionar_accion(estado)
    estado, r, terminado = env2.step(accion)
    ruta.append(estado)
    if terminado: break

print(f"\nRuta del agente entrenado: {' → '.join(str(s) for s in ruta)}")
print(f"¿Llegó a la meta? {'Sí ✓' if estado == (env.n-1, env.n-1) else 'No ✗'}")

## ✏️ Ejercicio 1: Double Q-Learning

**Problema:** El Q-Learning estándar sobreestima los valores Q porque usa
el mismo Q-table para seleccionar Y evaluar. Double Q-Learning usa dos tablas.

In [ ]:
# ✏️ EJERCICIO 1

class DoubleQAgent:
    """
    Double Q-Learning: dos tablas Q_A y Q_B.
    - Para seleccionar: usa Q_A para elegir la acción
    - Para evaluar: usa Q_B para obtener el valor
    - En cada paso, actualiza aleatoriamente Q_A o Q_B
    """
    def __init__(self, n_acciones=4, alpha=0.1, gamma=0.9, epsilon=0.8):
        self.n_acciones = n_acciones
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.q_a = {}   # Primera tabla
        self.q_b = {}   # Segunda tabla
    
    def _get_q(self, tabla, estado):
        if estado not in tabla:
            tabla[estado] = [0.0] * self.n_acciones
        return tabla[estado]
    
    def seleccionar_accion(self, estado):
        # TODO: Usar promedio de Q_A + Q_B para selección
        pass
    
    def aprender(self, estado, accion, recompensa, siguiente, terminado):
        # TODO: Con prob 0.5, actualizar Q_A usando Q_B para evaluar
        #       Con prob 0.5, actualizar Q_B usando Q_A para evaluar
        pass

# Test
double_agent = DoubleQAgent()
doble_rewards = entrenar(GridWorldEnv(semilla=42), double_agent, episodios=500)
print("Double Q-Learning — promedio últimos 100 episodios:",
      sum(doble_rewards[-100:]) / 100)

In [ ]:
# 💡 SOLUCIÓN — Ejercicio 1

class DoubleQAgent:
    def __init__(self, n_acciones=4, alpha=0.1, gamma=0.9, epsilon=0.8):
        self.n_acciones = n_acciones
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.q_a = {}
        self.q_b = {}
    
    def _get_q(self, tabla, estado):
        if estado not in tabla:
            tabla[estado] = [0.0] * self.n_acciones
        return tabla[estado]
    
    def seleccionar_accion(self, estado):
        if random.random() < self.epsilon:
            return random.randrange(self.n_acciones)
        # Suma de ambas tablas para selección
        qa = self._get_q(self.q_a, estado)
        qb = self._get_q(self.q_b, estado)
        combined = [a + b for a, b in zip(qa, qb)]
        return combined.index(max(combined))
    
    def aprender(self, estado, accion, recompensa, siguiente, terminado):
        if random.random() < 0.5:
            # Actualizar Q_A, evaluar con Q_B
            q_act = self._get_q(self.q_a, estado)[accion]
            mejor_accion = self._get_q(self.q_a, siguiente).index(max(self._get_q(self.q_a, siguiente)))
            max_q_sig = 0.0 if terminado else self._get_q(self.q_b, siguiente)[mejor_accion]
            target = recompensa + self.gamma * max_q_sig
            self._get_q(self.q_a, estado)[accion] += self.alpha * (target - q_act)
        else:
            # Actualizar Q_B, evaluar con Q_A
            q_act = self._get_q(self.q_b, estado)[accion]
            mejor_accion = self._get_q(self.q_b, siguiente).index(max(self._get_q(self.q_b, siguiente)))
            max_q_sig = 0.0 if terminado else self._get_q(self.q_a, siguiente)[mejor_accion]
            target = recompensa + self.gamma * max_q_sig
            self._get_q(self.q_b, estado)[accion] += self.alpha * (target - q_act)

env_d = GridWorldEnv(semilla=42)
double_agent = DoubleQAgent(epsilon=0.8)
for ep in range(1000):
    s = env_d.reset()
    done = False
    while not done:
        a = double_agent.seleccionar_accion(s)
        ns, r, done = env_d.step(a)
        double_agent.aprender(s, a, r, ns, done)
        s = ns
    double_agent.epsilon = max(0.01, double_agent.epsilon * 0.995)
print("Double Q-Learning entrenado. ✓")

## 🎯 Quiz — Preguntas de Entrevista

**P1:** ¿Cuál es la diferencia entre on-policy y off-policy learning?
> **R:** On-policy (SARSA) aprende el valor de la política que ESTÁ siguiendo.
> Off-policy (Q-Learning) aprende el valor de la política ÓPTIMA independientemente
> de cómo explora. Q-Learning converge a la política óptima aunque explore aleatoriamente.

**P2:** ¿Qué es el "exploration-exploitation tradeoff"?
> **R:** Explorar (acciones aleatorias) descubre nuevo conocimiento pero no maximiza recompensa inmediata.
> Explotar (mejor acción conocida) maximiza recompensa pero puede perder óptimos globales.
> Epsilon-greedy es el balance más simple; UCB y Thompson Sampling son más sofisticados.

**P3:** ¿Por qué el Q-Learning no escala bien a espacios de estados grandes?
> **R:** La Q-table tiene |S| × |A| entradas. Para Atari con estados de imagen 84×84×4,
> sería imposible. Solución: Deep Q-Network (DQN) usa una red neuronal para aproximar Q.

**P4:** ¿Qué es el "reward shaping" y cuándo es peligroso?
> **R:** Modificar la función de recompensa para aprendizaje más rápido.
> Peligroso cuando el agente aprende a maximizar la recompensa moldeada en lugar
> del objetivo real (reward hacking). Ejemplo: robot que da vueltas para acumular pequeñas recompensas.